# ForestWatch Papua — Training Spesifik Wilayah **Merauke + Boven Digoel**
### Model 1: Attention U-Net (SCSE) + ResNet-50 encoder (ImageNet)

Notebook **mandiri & end-to-end** untuk satu wilayah krisis deforestasi: *preprocessing →
EDA distribusi piksel (teks) → bundling → training → evaluasi → ONNX*. Semua data & artefak
disimpan di satu folder Drive khusus (`Training_Merauke_Boven_Digoel/`).

**Kenapa Merauke + Boven Digoel?** (data 2024–2025)
- **Merauke PSN food/sugarcane estate = driver deforestasi primer #1 Indonesia 2024**
  (22.272 ha ekosistem alami dibuka Jan'24–Jun'25; 9.835 ha hutan primer).
- **Boven Digoel** = hotspot sawit proyek Tanah Merah (≥3.700 ha hutan primer hilang).
- Kedua kabupaten **padat sinyal transisi Hutan→Lahan Terbuka / Sawit / Pertanian** →
  ideal untuk model belajar deforestasi pada dataset terfokus & kecil.

**Kenapa Attention U-Net (SCSE)?** (dukungan literatur)
- Attention gate pada skip-connection menekan aktivasi tak relevan & menonjolkan region
  informatif — penting untuk citra Sentinel-2 yang spektralnya tumpang-tindih antar-kelas.
- Blok **spatial-and-channel Squeeze-and-Excitation (scSE)** merekalibrasi fitur spasial &
  spektral secara adaptif.
- Brown et al. (2022), *"An attention-based U-Net for detecting deforestation within satellite
  sensor imagery"* (ISPRS / Int. J. Applied Earth Obs.) melaporkan F1 piksel 0.946–0.977 untuk
  deteksi deforestasi Sentinel-2 dengan Attention U-Net.
- *Attention-Based Semantic Segmentation Networks for Forest Applications* (Forests, 2023, MDPI)
  menegaskan keunggulan mekanisme attention untuk monitoring deforestasi berbasis penginderaan jauh.

**Cakupan tile (grid 6×6 atas bbox Papua):** `{24, 25, 26, 30, 31, 32}` — kolom lon 137.5–141.2,
lat -9.5 to -4.5 (Merauke + Boven Digoel; `idx = i*6 + j`, i=kolom lon, j=baris lat).

> **Catatan kejujuran data:** laporkan mIoU/akurasi **apa adanya** untuk wilayah ini. Lebih baik
> mIoU 0,68 yang nyata daripada 0,90 yang fiktif (sesuai pedoman ForestWatch & kriteria Substansi & Data).

## Bagian 0 — Setup environment (Colab / Lab)

- **Google Colab** → `ENV = "colab"`: clone repo + install package + mount Drive.
- **PC lab + Drive Desktop (mode Mirror)** → `ENV = "lab"`: sesuaikan `DRIVE_ROOT`, jalankan
  `pip install -e ".[ml]"` sekali di clone repo lokal.

Prasyarat: patch Papua se-36-tile sudah ada di `ForestWatch_Patches/tile_XXX/` di Drive (hasil
Bagian 1–14 notebook pipeline utama).

In [3]:
# === Bagian 0 — Setup (set ENV = "colab" / "lab") ===
ENV = "colab"   # "colab" (Google Colab) | "lab" (PC + Drive Desktop mode Mirror)
from pathlib import Path

DRIVE_ROOT = None

if ENV == "colab":
    import subprocess, sys, importlib
    subprocess.run(
        "cd /content && (git -C fw_repo pull -q || git clone --depth 1 "
        "https://github.com/Ridho-Dwi-Syahputra/forestwatch-model.git fw_repo)",
        shell=True, check=False,
    )
    subprocess.run("pip install -q -e /content/fw_repo[gee,gis,ml]", shell=True, check=False)
    if "/content/fw_repo/src" not in sys.path:
        sys.path.insert(0, "/content/fw_repo/src")
    for _m in [m for m in list(sys.modules) if m == "forestwatch" or m.startswith("forestwatch.")]:
        del sys.modules[_m]
    importlib.invalidate_caches()
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/Satria Data 3.0")
elif ENV == "lab":
    DRIVE_ROOT = Path(r"G:/My Drive/Satria Data 3.0")   # <- SESUAIKAN path mount Drive Desktop lab
else:
    raise ValueError("ENV harus 'colab' atau 'lab'")

assert DRIVE_ROOT.exists(), f"DRIVE_ROOT {DRIVE_ROOT} tidak ada — cek mount Drive."

import torch
_gpu = f" ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else ""
print(f"ENV={ENV} | DRIVE_ROOT={DRIVE_ROOT} | CUDA={torch.cuda.is_available()}{_gpu}")

Mounted at /content/drive
ENV=colab | DRIVE_ROOT=/content/drive/MyDrive/Satria Data 3.0 | CUDA=False


In [4]:
# === Path sumber + folder khusus wilayah + konfigurasi tile ===
import json
from forestwatch.config import load_config
from forestwatch.constants import N_CLASSES, CLASS_NAMES
from forestwatch.utils.io import save_json, load_json
cfg = load_config()

# --- Sumber patch (se-Papua, 36 tile) ---
PATCH_DIR         = DRIVE_ROOT / 'ForestWatch_Patches'            # papua/tile_XXX/p*.npz
PATCHES_TRANSFER  = DRIVE_ROOT / 'ForestWatch_Patches_Transfer'   # kelas langka dari wilayah lain
AUGMENTED_PATCHES = DRIVE_ROOT / 'Augmented_Patches'              # augmentasi offline kelas minor

# --- Folder khusus wilayah: SEMUA data & artefak Merauke+BD disimpan di sini ---
AREA_DIR  = DRIVE_ROOT / 'Training_Merauke_Boven_Digoel'
EDA_DIR   = AREA_DIR / 'eda_cache'
MODEL_DIR = AREA_DIR / 'model_1_attention_unet'
OUTPUT_DIR = MODEL_DIR / 'output'   # khusus GAMBAR (kurva, confusion, prediksi kualitatif)
for d in (AREA_DIR, EDA_DIR, MODEL_DIR, OUTPUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# --- Whitelist tile Merauke + Boven Digoel (grid 6x6, idx = i*6 + j) ---
TILE_WHITELIST = {24, 25, 26, 30, 31, 32}

print('AREA_DIR       :', AREA_DIR)
print('TILE_WHITELIST :', sorted(TILE_WHITELIST))
print('Resep training :', cfg['training']['loss']['type'],
      '| epochs:', cfg['training']['epochs'], '| batch:', cfg['training']['batch_size'])

AREA_DIR       : /content/drive/MyDrive/Satria Data 3.0/Training_Merauke_Boven_Digoel
TILE_WHITELIST : [24, 25, 26, 30, 31, 32]
Resep training : focal_tversky | epochs: 80 | batch: 8


## Bagian Tambahan — Ekspor GEE: Raja Ampat (sinyal **Tambang** asli)

Wilayah Merauke + Boven Digoel hampir tidak punya sinyal Tambang asli (drivernya sawit/food
estate, bukan tambang). Untuk memberi model contoh **Tambang yang nyata** (bukan cuma dari
patch transfer wilayah lain), kita ekspor 3 pulau tambang nikel aktif di **Raja Ampat**
(Papua Barat Daya) — **di LUAR bbox Papua yang sudah diekspor** (`130.0–141.2°E`), jadi
butuh ekspor GEE baru.

| Pulau | Status koordinat | Operator |
|---|---|---|
| **Gag** | ✅ presisi (-0.448, 129.885) | PT Gag Nikel — 623 ha terdampak 2001–2024 |
| **Kawe** | ⚠️ perkiraan distrik (Waigeo Barat) | PT KSM — 5.922 ha konsesi |
| **Manuran** | ⚠️ perkiraan distrik (utara Waigeo) | PT ASP — 1.173 ha konsesi |

> **PENTING:** Kawe & Manuran pakai bbox perkiraan (level distrik, BUKAN titik GPS presisi).
> Cell preview di bawah **WAJIB dijalankan dan dicek visual dulu** sebelum submit task
> export (supaya tidak buang kuota GEE untuk bbox yang salah/laut kosong). Kalau preview
> menunjukkan lokasi salah, sesuaikan bbox di `RAJA_AMPAT_REGIONS` lalu preview ulang.

In [5]:
# === Setup region Raja Ampat + folder ekspor ===
# Kawe & Manuran DIKELUARKAN: cek histogram label (cell berikut) menunjukkan kelas
# Tambang = 0 piksel di KETIGA pulau -- footprint Tang&Werner/Maus tidak cover Raja Ampat
# sama sekali (bukan soal bbox/vintage). Gag dipertahankan + override manual (cell baru di
# bawah) krn tambangnya jelas terlihat di RGB & terdokumentasi 623 ha (PT Gag Nikel).
RAJA_AMPAT_REGIONS = {
    'gag': (129.74, -0.58, 130.03, -0.30),   # presisi -- CONFIRMED preview RGB
}

TILES_RAJA_AMPAT   = DRIVE_ROOT / 'ForestWatch_Tiles_RajaAmpat'
PATCHES_RAJA_AMPAT = DRIVE_ROOT / 'ForestWatch_Patches_RajaAmpat'
for d in (TILES_RAJA_AMPAT, PATCHES_RAJA_AMPAT):
    d.mkdir(parents=True, exist_ok=True)

import ee
from forestwatch.gee.auth import init_ee
init_ee(project=cfg['project']['gee_project_id'])
RA_YEAR = cfg['periods']['t2']   # 2025 -- citra & label terbaru (deforestasi tambang aktif)
print(f"Region Raja Ampat: {list(RAJA_AMPAT_REGIONS)} | tahun citra/label: {RA_YEAR}")

2026-06-18 18:17:34 [WARNING] forestwatch.gee.auth: Init GEE gagal: Please authorize access to your Earth Engine account by running

earthengine authenticate

in your command line, or ee.Authenticate() in Python, and then retry.. Mencoba ee.Authenticate()...
2026-06-18 18:17:47 [INFO] forestwatch.gee.auth: GEE siap setelah re-auth (project=forestwatch-papua-unand).
Region Raja Ampat: ['gag'] | tahun citra/label: 2025


In [6]:
# === PREVIEW (WAJIB) -- cek visual sebelum submit task export GEE ===
from forestwatch.gee.composite import s2_composite
from IPython.display import Image as IPyImage, display

for name, bbox in RAJA_AMPAT_REGIONS.items():
    box = ee.Geometry.Rectangle(list(bbox))
    img = s2_composite(RA_YEAR, box)
    # s2_composite() sudah membagi ke reflektansi [0,1] (bukan DN mentah 0-10000) ->
    # stretch yang benar min=0, max=0.3 (konsisten dgn notebook utama, lihat
    # forestwatch_papua_full_pipeline.ipynb Map2.addLayer stack_t2 RGB).
    vis = img.visualize(bands=['B4', 'B3', 'B2'], min=0, max=0.3)
    url = vis.getThumbURL({'region': box, 'dimensions': 512, 'format': 'png'})
    print(f"{name}  bbox={bbox}")
    display(IPyImage(url=url))

print("\nCek: apakah terlihat pulau + area gundul/coklat (bekas tambang) di tengah hijau hutan?")
print("Kalau salah satu region terlihat cuma laut kosong, perbaiki bbox di RAJA_AMPAT_REGIONS lalu re-run cell ini.")

gag  bbox=(129.74, -0.58, 130.03, -0.3)



Cek: apakah terlihat pulau + area gundul/coklat (bekas tambang) di tengah hijau hutan?
Kalau salah satu region terlihat cuma laut kosong, perbaiki bbox di RAJA_AMPAT_REGIONS lalu re-run cell ini.


In [7]:
# === Override manual Tambang utk Gag -- footprint global tidak cover pulau ini ===
# Strategi: kelas Tambang TIDAK bisa didapat dari overlay poligon (sudah terbukti 0 px).
# Sebagai gantinya, deteksi semi-otomatis: piksel non-vegetasi (NDVI rendah) DAN bukan air
# (sudah diklasifikasi DW base) DAN berada dalam sub-bbox konsesi PT Gag Nikel (blok tengah
# pulau, deskripsi publik: "central block extending west-east coast, ~1/3 pulau, arah
# tenggara"). WAJIB cek preview overlay di bawah sebelum dipakai -- sub-bbox ini estimasi,
# bukan survei presisi.
from forestwatch.gee.label_fusion import build_label

GAG_MINING_SUBBBOX = (129.82, -0.50, 129.96, -0.38)   # sub-area konsesi -- VERIFIKASI overlay
NDVI_BARE_THRESHOLD = 0.25                             # < ini = non-vegetasi (tanah terbuka/tambang)

def gag_tambang_override(label_img, composite_img):
    # Mask piksel yang dipaksa jadi kelas 5 (Tambang) dlm sub-bbox konsesi Gag.
    sub_box = ee.Geometry.Rectangle(list(GAG_MINING_SUBBBOX))
    ndvi = composite_img.normalizedDifference(['B8', 'B4'])
    is_bare = ndvi.lt(NDVI_BARE_THRESHOLD)
    is_not_water = label_img.neq(0)
    is_in_subbbox = ee.Image.constant(1).clip(sub_box).mask().unmask(0)
    return is_bare.And(is_not_water).And(is_in_subbbox)

# --- Preview overlay (WAJIB cek sebelum lanjut) ---
_gag_box  = ee.Geometry.Rectangle(list(RAJA_AMPAT_REGIONS['gag']))
_gag_img  = s2_composite(RA_YEAR, _gag_box)
_gag_lbl  = build_label(_gag_box, RA_YEAR, composite=_gag_img)
_gag_mask = gag_tambang_override(_gag_lbl, _gag_img)

_vis_rgb  = _gag_img.visualize(bands=['B4', 'B3', 'B2'], min=0, max=0.3)
_vis_mask = _gag_mask.selfMask().visualize(palette=['ff00ff'])   # magenta = area override
_overlay  = ee.ImageCollection([_vis_rgb, _vis_mask]).mosaic()
_url = _overlay.getThumbURL({'region': _gag_box, 'dimensions': 512, 'format': 'png'})
print("Overlay magenta = piksel yang akan DIPAKSA jadi Tambang.")
print("Cek: apakah magenta nutupin bercak coklat/gundul yang sama dgn preview RGB sebelumnya?")
print("Kalau magenta meleset (kena laut/hutan lain), sesuaikan GAG_MINING_SUBBBOX lalu re-run.")
display(IPyImage(url=_url))

Overlay magenta = piksel yang akan DIPAKSA jadi Tambang.
Cek: apakah magenta nutupin bercak coklat/gundul yang sama dgn preview RGB sebelumnya?
Kalau magenta meleset (kena laut/hutan lain), sesuaikan GAG_MINING_SUBBBOX lalu re-run.


In [8]:
# === VERIFIKASI LABEL (lebih penting dari RGB!) -- cek apakah kelas Tambang (5) BENAR
# muncul di region ini. Berbeda dari preview RGB di atas: label Tambang TIDAK dideteksi
# dari piksel citra, murni overlay POLIGON STATIS Tang & Werner (2023) + Maus dkk. (2022).
# Tambang Kawe/Manuran baru ramai 2024-2025 (LEBIH BARU dari vintage dataset) -> berisiko
# pixel mining scar yang terlihat di RGB malah berlabel Hutan/Lahan Terbuka, BUKAN Tambang.
from forestwatch.gee.label_fusion import build_label
from forestwatch.constants import CLASS_NAMES

print("Distribusi label per region (cek kelas 5=Tambang benar2 ada / tidak):\n")
for name, bbox in RAJA_AMPAT_REGIONS.items():
    box = ee.Geometry.Rectangle(list(bbox))
    img = s2_composite(RA_YEAR, box)
    label = build_label(box, RA_YEAR, composite=img)
    if name == 'gag':
        label = label.where(gag_tambang_override(label, img), 5)
    hist = label.reduceRegion(
        reducer=ee.Reducer.frequencyHistogram(), geometry=box, scale=10, maxPixels=1e9,
    ).get('label').getInfo()
    total = sum(hist.values()) or 1
    print(f"{name}:")
    for cls_str, cnt in sorted(hist.items(), key=lambda x: int(x[0])):
        cls = int(cls_str)
        print(f"  kelas {cls} ({CLASS_NAMES[cls]:<14}): {cnt:>12,.0f} px ({100*cnt/total:5.1f}%)")
    tambang_px = hist.get('5', 0)
    if tambang_px == 0:
        print(f"  >>> PERHATIAN: kelas Tambang (5) = 0 piksel! Footprint Tang&Werner/Maus "
              f"belum cover '{name}' -- region ini TIDAK akan menyumbang sinyal Tambang asli "
              f"meski tambangnya terlihat di RGB.")
    print()

Distribusi label per region (cek kelas 5=Tambang benar2 ada / tidak):

gag:
  kelas 0 (Perairan      ):    9,451,249 px ( 93.9%)
  kelas 1 (Hutan         ):      522,252 px (  5.2%)
  kelas 2 (Lahan Terbuka ):       18,970 px (  0.2%)
  kelas 3 (Sawit         ):        2,151 px (  0.0%)
  kelas 4 (Pertanian Lain):       36,486 px (  0.4%)
  kelas 5 (Tambang       ):       27,388 px (  0.3%)
  kelas 6 (Permukiman    ):        3,581 px (  0.0%)



In [ ]:
# === Submit task ekspor GEE (citra + label 7-kelas) -- jalankan SETELAH preview OK ===
from forestwatch.gee.label_fusion import build_label
from forestwatch.gee.tiles import make_tiles
from forestwatch.gee.export import export_tiles_grid

NX_RA, NY_RA = 2, 2   # grid kecil per pulau (bbox sudah kecil)

raja_ampat_tasks = []
for name, bbox in RAJA_AMPAT_REGIONS.items():
    box = ee.Geometry.Rectangle(list(bbox))
    img = s2_composite(RA_YEAR, box)
    label = build_label(box, RA_YEAR, composite=img)
    if name == 'gag':
        label = label.where(gag_tambang_override(label, img), 5)
    stack = img.addBands(label.toFloat())
    tiles = make_tiles(box, nx=NX_RA, ny=NY_RA)
    tasks = export_tiles_grid(
        stack, tiles, name_prefix=f'rajaampat_{name}_tile',
        folder=f'ForestWatch_Tiles_RajaAmpat/{name}',
        scale=cfg['sentinel2']['scale'], max_pixels=int(cfg['export']['max_pixels']),
    )
    raja_ampat_tasks.extend(tasks)

print(f"{len(raja_ampat_tasks)} task disubmit -> ForestWatch_Tiles_RajaAmpat/<pulau>/")
print("Pantau: https://code.earthengine.google.com/tasks")
print("Setelah SEMUA task COMPLETED, lanjut ke cell berikut (cut patches).")

In [ ]:
# === (Opsional) Monitor task sampai selesai -- blocking, polling tiap 60s ===
from forestwatch.gee.export import monitor_tasks

final_states = monitor_tasks(raja_ampat_tasks, poll_seconds=60, timeout_seconds=3*3600)
print('Status akhir:', final_states)

In [9]:
# === Cut patches dari tile Raja Ampat yang sudah selesai diekspor (cached/idempoten) ===
from forestwatch.data.patches import cut_patches, list_patches

raja_ampat_files = []
for name in RAJA_AMPAT_REGIONS:
    tile_dir  = TILES_RAJA_AMPAT / name
    patch_dir = PATCHES_RAJA_AMPAT / name
    n_tif = len(sorted(tile_dir.glob('*.tif'))) if tile_dir.exists() else 0
    if n_tif == 0:
        print(f"  [skip] {name}: belum ada .tif di {tile_dir} (task export belum selesai?)")
        continue
    existing = list_patches(patch_dir)
    if existing:
        print(f"  [cache] {name}: {len(existing)} patch sudah ada, skip cut ulang.")
    else:
        cut_patches(tile_dir, patch_dir, patch_size=cfg['patches']['size'],
                    stride=cfg['patches']['size'])
        existing = list_patches(patch_dir)
        print(f"  [ok] {name}: {n_tif} tif -> {len(existing)} patch")
    raja_ampat_files.extend(existing)

print(f"\nTotal patch Raja Ampat: {len(raja_ampat_files):,}")
if not raja_ampat_files:
    print("PERHATIAN: raja_ampat_files kosong -- cell filter berikutnya akan lanjut TANPA Tambang asli.")

  [cache] gag: 144 patch sudah ada, skip cut ulang.

Total patch Raja Ampat: 144


In [10]:
# === Preprocessing 1: filter geografis Merauke+BD + split holdout ===
import numpy as np
from forestwatch.data import list_patches, split_files

def _tile_idx(f):
    # PATCH_DIR/tile_030/p*.npz -> 30 (atau -1 bila bukan subfolder tile)
    name = Path(f).parent.name
    return int(name.split("_")[-1]) if name.startswith("tile_") else -1

# Patch Papua DIFILTER ke wilayah; transfer + aug DIPERTAHANKAN (bantu kelas langka di TRAIN saja).
papua_all      = list_patches(PATCH_DIR)
papua_files    = [f for f in papua_all if _tile_idx(f) in TILE_WHITELIST]
# Transfer (kelas langka wilayah LAIN): disubsample agar train ramping (Merauke+BD fokus
# sawit/pertanian; tambang nyaris absen -> tak butuh semua transfer). Deterministik seed=42.
import random as _rnd
TRANSFER_FRACTION = 0.40
# Tambang LOKAL Merauke+BD nyaris absen (lihat VALIDASI di bawah: val/test holdout murni
# cuma dapat ~1 patch Tambang dari Gag override -- terlalu kecil utk IoU yang reliable).
# Solusi: sisihkan SEBAGIAN patch transfer/tambang (region established: Sangatta, Morowali,
# dll -- BUKAN representasi prevalensi lokal Merauke+BD) sebagai test SUPLEMEN khusus
# Tambang, terpisah dari holdout utama. Disisihkan SEBELUM sampling TRANSFER_FRACTION agar
# tidak ikut terpakai di train (no leakage).
TAMBANG_SUPPLEMENTARY_FRACTION = 0.15
_tambang_transfer_all = list_patches(PATCHES_TRANSFER / 'tambang')
_tb_shuffled = _tambang_transfer_all.copy()
_rnd.Random(43).shuffle(_tb_shuffled)
_n_tb_holdout = int(len(_tb_shuffled) * TAMBANG_SUPPLEMENTARY_FRACTION)
tambang_supplementary_test = _tb_shuffled[:_n_tb_holdout]
_tambang_holdout_set = {str(p) for p in tambang_supplementary_test}
print(f'Tambang supplementary test (dari transfer, BUKAN Merauke+BD): '
      f'{len(tambang_supplementary_test):,} / {len(_tambang_transfer_all):,} patch transfer/tambang')

_transfer_all  = [f for f in list_patches(PATCHES_TRANSFER) if str(f) not in _tambang_holdout_set]
transfer_files = (_rnd.Random(42).sample(_transfer_all, int(len(_transfer_all) * TRANSFER_FRACTION))
                  if 0 < TRANSFER_FRACTION < 1.0 else _transfer_all)
print(f'transfer disubsample: {len(_transfer_all):,} -> {len(transfer_files):,} (fraction={TRANSFER_FRACTION})')

# aug_files = hasil flip/rotate/brightness dari source PATCHES_TRANSFER (lihat
# scripts/augment_offline.py: parent=f.name disimpan per file augmentasi). Patch
# 'tambang' yang sourcenya masuk tambang_supplementary_test HARUS dikeluarkan juga
# di sini -- kalau tidak, model tetap "melihat" lokasi itu (versi flip/rotate) lewat
# train meski source aslinya dipakai sbg test (leakage).
_aug_all = list_patches(AUGMENTED_PATCHES)
_tambang_holdout_names = {Path(p).name for p in tambang_supplementary_test}

def _aug_is_leak(f):
    if Path(f).parent.name != 'tambang' or not _tambang_holdout_names:
        return False
    try:
        return str(np.load(f)['parent']) in _tambang_holdout_names
    except Exception:
        return False

aug_files = [f for f in _aug_all if not _aug_is_leak(f)]
_n_leak = len(_aug_all) - len(aug_files)
if _n_leak:
    print(f'[anti-leak] {_n_leak} aug_files (derivatif tambang_supplementary_test) dikeluarkan dari train.')
assert papua_files, (
    f"Tidak ada patch Merauke+BD di {PATCH_DIR} untuk tile {sorted(TILE_WHITELIST)}. "
    "Cek nama subfolder tile_XXX & sync Drive."
)

# Distribusi per-tile (transparansi cakupan)
from collections import Counter
_per_tile = Counter(_tile_idx(f) for f in papua_files)
print("Patch papua per-tile (Merauke + Boven Digoel):")
for ti in sorted(TILE_WHITELIST):
    print(f"  tile_{ti:03d}: {_per_tile.get(ti, 0):>7,} patch")

# Split sumber-aware (seed=42): val/test = holdout MURNI Merauke+BD; train + transfer + aug.
train_p, val_p, test_p = split_files(papua_files, train_ratio=0.8, val_ratio=0.1, seed=42)

# Raja Ampat (Tambang asli, jika sudah diekspor & dipotong patch di section sebelumnya):
# split TERPISAH (sumber beda) dgn rasio sama -> sebagian masuk val/test agar IoU Tambang
# punya holdout NYATA, bukan cuma dari transfer.
_ra_files = globals().get('raja_ampat_files', [])
if _ra_files:
    ra_train, ra_val, ra_test = split_files(_ra_files, train_ratio=0.8, val_ratio=0.1, seed=42)
    train_p = list(train_p) + list(ra_train)
    val_p   = list(val_p) + list(ra_val)
    test_p  = list(test_p) + list(ra_test)
    print(f'+ Raja Ampat: train={len(ra_train)} val={len(ra_val)} test={len(ra_test)}')
else:
    print('Raja Ampat belum ada / dilewati -- holdout Tambang hanya dari transfer (jika ada).')

final_train_files = list(train_p) + list(transfer_files) + list(aug_files)
print(f"\npapua(Merauke+BD)={len(papua_files):,} dari {len(papua_all):,} total se-Papua")
print(f"train={len(final_train_files):,} (papua={len(train_p):,}+transfer={len(transfer_files):,}"
      f"+aug={len(aug_files):,}) | val={len(val_p):,} | test={len(test_p):,} (holdout Merauke+BD)")

Tambang supplementary test (dari transfer, BUKAN Merauke+BD): 1,444 / 9,632 patch transfer/tambang
transfer disubsample: 12,848 -> 5,139 (fraction=0.4)
[anti-leak] 883 aug_files (derivatif tambang_supplementary_test) dikeluarkan dari train.
Patch papua per-tile (Merauke + Boven Digoel):
  tile_024:   2,401 patch
  tile_025:   1,568 patch
  tile_026:   1,127 patch
  tile_030:   1,127 patch
  tile_031:     736 patch
  tile_032:   2,401 patch
+ Raja Ampat: train=115 val=14 test=15

papua(Merauke+BD)=9,360 dari 193,453 total se-Papua
train=15,883 (papua=7,603+transfer=5,139+aug=3,141) | val=950 | test=951 (holdout Merauke+BD)


In [12]:
# === Preprocessing 2: scan distribusi piksel per patch (cached -> skip jika sudah ada) ===
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

SCAN_CACHE = EDA_DIR / 'patch_class_counts.json'

def _count_patch(path):
    lab = np.load(path)['lab'].flatten().astype(int)
    return np.bincount(lab, minlength=N_CLASSES).tolist()

def scan_patches(files, cache_path, max_workers=64):
    cache_p = Path(cache_path)
    dist = {}
    if cache_p.exists():
        dist = json.loads(cache_p.read_text(encoding='utf-8'))
        missing = [f for f in files if str(f) not in dist]
        if not missing:
            print(f"[cache] {cache_p.name}: semua {len(files):,} patch sudah tercache.")
            return dist
        print(f"[cache] {len(missing):,} patch baru, scan sekarang ...")
        files = missing
    with ThreadPoolExecutor(max_workers=max_workers) as exe:
        futs = {exe.submit(_count_patch, f): str(f) for f in files}
        for fut in tqdm(as_completed(futs), total=len(futs), desc='Scan piksel', unit='patch'):
            dist[futs[fut]] = fut.result()
    cache_p.write_text(json.dumps(dist), encoding='utf-8')
    print(f"[cache] disimpan -> {cache_p}")
    return dist

# Scan SEMUA file yang dipakai (train final + val + test) sekali, simpan di folder wilayah.
patch_dist = scan_patches(list(final_train_files) + list(val_p) + list(test_p), SCAN_CACHE)
print(f"Total patch tercatat: {len(patch_dist):,}")

[cache] patch_class_counts.json: semua 17,784 patch sudah tercache.
Total patch tercatat: 21,405


In [13]:
# === VALIDASI distribusi PATCH & PIKSEL per 7 kelas (train / val / test) ===
def _stats(files):
    px   = [0] * N_CLASSES   # total piksel per kelas
    pres = [0] * N_CLASSES   # jumlah patch yang MENGANDUNG kelas (>= 1 px)
    dom  = [0] * N_CLASSES   # jumlah patch dgn kelas dominan
    for f in files:
        c = patch_dist.get(str(f), [0] * N_CLASSES)
        for k in range(N_CLASSES):
            px[k] += c[k]
            if c[k] > 0:
                pres[k] += 1
        dom[int(np.argmax(c))] += 1
    return px, pres, dom

def validasi(files, title):
    px, pres, dom = _stats(files)
    grand = sum(px) or 1
    npat = len(files) or 1
    print('=' * 88)
    print(f"  {title}  |  {len(files):,} patch  |  {grand:,} piksel")
    print('=' * 88)
    print(f"  {'Kelas':<16}{'Piksel':>16}{'Px %':>8}{'Patch berisi':>14}{'(%)':>8}{'Patch dominan':>15}")
    print('  ' + '-' * 84)
    for c in range(N_CLASSES):
        print(f"  {CLASS_NAMES[c]:<16}{px[c]:>16,}{100*px[c]/grand:>7.2f}%"
              f"{pres[c]:>14,}{100*pres[c]/npat:>7.1f}%{dom[c]:>15,}")
    print('  ' + '-' * 84)
    print(f"  {'TOTAL':<16}{grand:>16,}{100.0:>7.2f}%{npat:>14,}{100.0:>7.1f}%{npat:>15,}")
    absent = [CLASS_NAMES[c] for c in range(N_CLASSES) if px[c] == 0]
    if absent:
        print(f"  PERHATIAN: kelas TANPA piksel di split ini -> {', '.join(absent)}")
    print('=' * 88)
    return px

print("VALIDASI DISTRIBUSI — 7 kelas x (train / val / test), wilayah Merauke + Boven Digoel\n")
px_train = validasi(final_train_files, "TRAIN FINAL (Merauke+BD + transfer + aug)")
px_val   = validasi(val_p,  "VAL  (holdout Merauke+BD murni)")
px_test  = validasi(test_p, "TEST (holdout Merauke+BD murni)")

# Interpretasi naratif (untuk penulisan esai)
_tr = sum(px_train) or 1
dom_c  = int(np.argmax(px_train))
rare_c = int(np.argmin([p if p > 0 else 10**18 for p in px_train]))
print("\nInterpretasi:")
print(f"- Dominan TRAIN  : {CLASS_NAMES[dom_c]} ({100*px_train[dom_c]/_tr:.1f}% piksel) — wajar (pesisir/hutan luas).")
print(f"- Terlangka TRAIN: {CLASS_NAMES[rare_c]} ({100*px_train[rare_c]/_tr:.2f}% piksel) — "
      "ditangani median-frequency class weights + weighted sampler.")
print("- Val/Test MURNI Merauke+BD (tanpa transfer/aug) -> metrik jujur per wilayah.")
print("- Kolom 'Patch berisi' = jumlah patch yang punya >=1 piksel kelas itu (cek keterwakilan).")
print("- Kelas absen di val/test -> IoU kelas itu tak terukur di holdout (catat di esai).")

VALIDASI DISTRIBUSI — 7 kelas x (train / val / test), wilayah Merauke + Boven Digoel

  TRAIN FINAL (Merauke+BD + transfer + aug)  |  15,883 patch  |  1,040,908,288 piksel
  Kelas                     Piksel    Px %  Patch berisi     (%)  Patch dominan
  ------------------------------------------------------------------------------------
  Perairan             551,112,899  52.95%        12,317   77.5%          8,392
  Hutan                218,694,852  21.01%         7,943   50.0%          3,684
  Lahan Terbuka         43,688,526   4.20%         6,949   43.8%            413
  Sawit                 40,524,920   3.89%         5,714   36.0%            562
  Pertanian Lain        86,698,532   8.33%         6,950   43.8%          1,148
  Tambang               16,161,715   1.55%           669    4.2%            257
  Permukiman            84,026,844   8.07%         5,019   31.6%          1,427
  ------------------------------------------------------------------------------------
  TOTAL       

In [14]:
# === Preprocessing 3: balanced subsampling -- cap kelas dominan relatif ke MEDIAN ===
# Berbeda dari versi sebelumnya (cap manual cuma Perairan/Hutan): di sini SEMUA kelas
# dicek. Kelas dgn total piksel > cap_ratio x median dipangkas (subsample acak hingga
# piksel turun ke target); kelas <= itu DIPERTAHANKAN PENUH (tidak menambah data palsu).
import random

SUBSAMPLE_ENABLED = True
CAP_RATIO = 2.0   # kelas dominan dipangkas sampai <= 2x median piksel-kelas. Naikkan
                   # (mis. 3.0) jika ingin lebih banyak data; turunkan utk lebih seimbang.

def balance_classes(files, cap_ratio=2.0, seed=42):
    rng = random.Random(seed)
    groups = {c: [] for c in range(N_CLASSES)}
    px_total = [0] * N_CLASSES
    for f in files:
        c = patch_dist.get(str(f), [0] * N_CLASSES)
        for k in range(N_CLASSES):
            px_total[k] += c[k]
        groups[int(np.argmax(c))].append(f)

    nz = [p for p in px_total if p > 0]
    median_px = sorted(nz)[len(nz) // 2] if nz else 0
    target = median_px * cap_ratio
    print(f"Median piksel/kelas (non-nol): {median_px:,}  |  cap target: {target:,.0f}\n")

    selected = []
    for cls in range(N_CLASSES):
        g = list(groups[cls])
        if px_total[cls] <= target or target == 0:
            selected.extend(g)
            print(f"  {CLASS_NAMES[cls]:<16}: keep semua {len(g):,} patch ({px_total[cls]:,} px)")
            continue
        rng.shuffle(g)
        tot, kept = 0, []
        for f in g:
            tot += patch_dist.get(str(f), [0] * N_CLASSES)[cls]
            kept.append(f)
            if tot >= target:
                break
        selected.extend(kept)
        print(f"  {CLASS_NAMES[cls]:<16}: {len(g):,} -> {len(kept):,} patch "
              f"({px_total[cls]:,} -> {tot:,} px)")
    return selected

if SUBSAMPLE_ENABLED:
    print("Balanced subsampling AKTIF (cap relatif median, semua 7 kelas):")
    selected_train = balance_classes(final_train_files, cap_ratio=CAP_RATIO)
    print(f"\nTRAIN: {len(final_train_files):,} -> {len(selected_train):,} patch")
    validasi(selected_train, "TRAIN setelah balanced subsampling")
else:
    selected_train = list(final_train_files)
    print(f"Subsample NONAKTIF -- pakai semua {len(selected_train):,} patch train apa adanya.")

Balanced subsampling AKTIF (cap relatif median, semua 7 kelas):
Median piksel/kelas (non-nol): 84,026,844  |  cap target: 168,053,688

  Perairan        : 8,392 -> 2,608 patch (551,112,899 -> 168,064,225 px)
  Hutan           : 3,684 -> 3,309 patch (218,694,852 -> 168,085,841 px)
  Lahan Terbuka   : keep semua 413 patch (43,688,526 px)
  Sawit           : keep semua 562 patch (40,524,920 px)
  Pertanian Lain  : keep semua 1,148 patch (86,698,532 px)
  Tambang         : keep semua 257 patch (16,161,715 px)
  Permukiman      : keep semua 1,427 patch (84,026,844 px)

TRAIN: 15,883 -> 9,724 patch
  TRAIN setelah balanced subsampling  |  9,724 patch  |  637,272,064 piksel
  Kelas                     Piksel    Px %  Patch berisi     (%)  Patch dominan
  ------------------------------------------------------------------------------------
  Perairan             177,371,121  27.83%         6,348   65.3%          2,608
  Hutan                196,236,156  30.79%         7,026   72.3%          3,3

In [15]:
# === Preprocessing 4: hitung class weights (median-frequency) dari TRAIN terpilih ===
from forestwatch.training.metrics import median_frequency_weights

dist_train = [0] * N_CLASSES
for f in selected_train:
    c = patch_dist.get(str(f), [0]*N_CLASSES)
    for k in range(N_CLASSES):
        dist_train[k] += c[k]

class_weights = median_frequency_weights(dict(enumerate(dist_train)), n_classes=N_CLASSES)
print("Class weights (median-frequency, dari TRAIN Merauke+BD):")
for c in range(N_CLASSES):
    print(f"  {CLASS_NAMES[c]:<16} piksel={dist_train[c]:>16,}  weight={class_weights[c]:>8.4f}")
save_json({'class_weights': [float(w) for w in class_weights]}, AREA_DIR / 'class_weights.json')
print(f"\nDisimpan -> {AREA_DIR / 'class_weights.json'}")

Class weights (median-frequency, dari TRAIN Merauke+BD):
  Perairan         piksel=     177,371,121  weight=  0.4590
  Hutan            piksel=     196,236,156  weight=  0.4150
  Lahan Terbuka    piksel=      41,805,388  weight=  1.9490
  Sawit            piksel=      39,166,088  weight=  2.0800
  Pertanian Lain   piksel=      85,251,718  weight=  0.9560
  Tambang          piksel=      15,969,206  weight=  5.0000
  Permukiman       piksel=      81,472,387  weight=  1.0000

Disimpan -> /content/drive/MyDrive/Satria Data 3.0/Training_Merauke_Boven_Digoel/class_weights.json


In [16]:
# === Preprocessing 5: bundle .tar + sampler cache ke folder wilayah (idempoten) ===
from forestwatch.data.dataset import create_dataset_archives, compute_patch_sampler_weights

def _arcname(f):
    f = Path(f)
    for root, pfx in [(PATCH_DIR, 'papua'), (PATCHES_TRANSFER, 'transfer'),
                      (AUGMENTED_PATCHES, 'aug'), (PATCHES_RAJA_AMPAT, 'rajaampat')]:
        try:
            return f"{pfx}/{f.relative_to(root).as_posix()}"
        except ValueError:
            continue
    return f.name

sel_set = set(_arcname(f) for f in selected_train)
train_items = [(_arcname(f), f) for f in selected_train]
val_items   = [(_arcname(f), f) for f in val_p]
test_items  = [(_arcname(f), f) for f in test_p]

splits = {'train': train_items, 'val': val_items, 'test': test_items}
create_dataset_archives(splits, AREA_DIR, n_train_parts=4, max_workers=64)

# Daftar arcname train terpilih (dipakai notebook lain / audit).
save_json(sorted(sel_set), AREA_DIR / 'selected_train_patches.json')

# Sampler cache SHARED — key = parts[-3:] dari arcname berprefiks 'train/' (cocok dgn key dari
# path lokal hasil extract_dataset_archives). Idempoten: skip bila sudah ada.
SAMPLER_CACHE = AREA_DIR / 'patch_sampler_weights_shared.json'
if SAMPLER_CACHE.exists():
    print(f"[skip] {SAMPLER_CACHE.name} sudah ada.")
else:
    sampler_files = [src for _, src in train_items]
    sampler_keys  = ["/".join((Path('train') / arc).parts[-3:]) for arc, _ in train_items]
    compute_patch_sampler_weights(sampler_files, class_weights, cache_path=SAMPLER_CACHE, keys=sampler_keys)
    print(f"[ok] {SAMPLER_CACHE.name} dihitung.")
print(f"\nSemua bahan training tersimpan di: {AREA_DIR}")

[skip] patch_sampler_weights_shared.json sudah ada.

Semua bahan training tersimpan di: /content/drive/MyDrive/Satria Data 3.0/Training_Merauke_Boven_Digoel


In [17]:
# === Preprocessing 6: ekstrak .tar ke disk LOKAL (I/O cepat, lepas dari Drive FUSE) ===
from forestwatch.data.dataset import extract_dataset_archives

LOCAL_DIR = (Path('/content/merauke_local') if ENV == 'colab' else Path.home() / 'merauke_local')
local_dirs = extract_dataset_archives(AREA_DIR, LOCAL_DIR, max_workers=64)

final_train_files = list_patches(local_dirs['train'])
val_files  = list_patches(local_dirs['val'])
test_files = list_patches(local_dirs['test'])
# Key sampler dari path lokal = parts[-3:] -> cocok dgn cache (sampler_keys=None aman).
print(f"[lokal] train={len(final_train_files):,} val={len(val_files):,} test={len(test_files):,} -> {LOCAL_DIR}")

Extract train_part01.tar:   0%|          | 0/2608 [00:00<?, ?file/s]

Extract train_part02.tar:   0%|          | 0/2608 [00:00<?, ?file/s]

Extract train_part03.tar:   0%|          | 0/2607 [00:00<?, ?file/s]

Extract train_part04.tar:   0%|          | 0/2607 [00:00<?, ?file/s]

Extract val_part01.tar:   0%|          | 0/950 [00:00<?, ?file/s]

Extract test_part01.tar:   0%|          | 0/951 [00:00<?, ?file/s]

[lokal] train=10,430 val=950 test=951 -> /content/merauke_local


In [ ]:
# === EDA cepat: distribusi piksel -- TRAIN FINAL (Merauke+BD+transfer+aug) vs Val/Test holdout ===
# Sanity-check sebelum training. Hasil di-cache ke EDA_DIR/pixel_dist_cache.json sehingga run
# berikutnya (pakai GPU) langsung baca JSON, tidak scan ulang patch dari Drive.
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm


def _count_pixels(files, max_workers=64):
    counts = {c: 0 for c in range(N_CLASSES)}
    if not files:
        return counts

    def _read_lab(f):
        return np.load(f)['lab']

    with ThreadPoolExecutor(max_workers=max_workers) as exe:
        for lab in tqdm(exe.map(_read_lab, files), total=len(files), desc='Counting pixels'):
            u, cnt = np.unique(lab, return_counts=True)
            for cls, n in zip(u.tolist(), cnt.tolist()):
                if 0 <= int(cls) < N_CLASSES:
                    counts[int(cls)] += int(n)
    return counts


# Cache pixel dist -- hasil deterministik untuk dataset tetap (skip scan ulang)
_dist_cache_path = EDA_DIR / 'pixel_dist_cache.json'
_dist_cache_path.parent.mkdir(parents=True, exist_ok=True)
_cache_key = (len(final_train_files), len(val_p), len(test_p))

_cached = None
if _dist_cache_path.exists():
    try:
        _c = json.loads(_dist_cache_path.read_text(encoding='utf-8'))
        if (_c.get('train_n'), _c.get('val_n'), _c.get('test_n')) == _cache_key:
            _cached = _c
            print(f'[cache] pixel_dist_cache.json dibaca '
                  f'(train={_c["train_n"]}, val={_c["val_n"]}, test={_c["test_n"]}) '
                  f'-- skip Counting pixels.')
    except Exception:
        _cached = None

if _cached:
    _train_dist = {int(k): v for k, v in _cached['train'].items()}
    _val_dist   = {int(k): v for k, v in _cached['val'].items()}
    _test_dist  = {int(k): v for k, v in _cached['test'].items()}
else:
    _train_dist = _count_pixels(final_train_files)
    _val_dist   = _count_pixels(val_p)
    _test_dist  = _count_pixels(test_p)
    _dist_cache_path.write_text(json.dumps({
        'train_n': _cache_key[0], 'val_n': _cache_key[1], 'test_n': _cache_key[2],
        'train': _train_dist, 'val': _val_dist, 'test': _test_dist,
    }), encoding='utf-8')
    print(f'[cache] pixel_dist_cache.json disimpan ke {_dist_cache_path}')

split_dist = {
    f'Train FINAL ({len(final_train_files)}p)': _train_dist,
    f'Val ({len(val_p)}p)':                     _val_dist,
    f'Test ({len(test_p)}p)':                   _test_dist,
}

print('Proporsi piksel per kelas -- Train FINAL (Merauke+BD+transfer+aug) vs Val/Test holdout:\n')
print(f"  {'Kelas':<16}" + ''.join(f'{name:>22}' for name in split_dist))
for c in range(N_CLASSES):
    row = f"  {CLASS_NAMES[c]:<16}"
    for d in split_dist.values():
        total = sum(d.values()) or 1
        row += f'{100 * d.get(c, 0) / total:5.1f}% ({d.get(c, 0):>13,})'.rjust(22)
    print(row)
print(f"  {'TOTAL piksel':<16}" + ''.join(f'{sum(d.values()):>22,}' for d in split_dist.values()))


Counting pixels:   0%|          | 0/10430 [00:00<?, ?it/s]

Counting pixels:   0%|          | 0/950 [00:00<?, ?it/s]

Counting pixels:   0%|          | 0/951 [00:00<?, ?it/s]

[cache] pixel_dist_cache.json disimpan ke /content/drive/MyDrive/Satria Data 3.0/Training_Merauke_Boven_Digoel/eda_cache/pixel_dist_cache.json
Proporsi piksel per kelas -- Train FINAL (Merauke+BD+transfer+aug) vs Val/Test holdout:

  Kelas             Train FINAL (10430p)            Val (950p)           Test (951p)
  Perairan         26.1% (  178,453,326) 99.3% (   61,795,524) 99.6% (   62,056,706)
  Hutan            29.3% (  200,415,698)  0.7% (      405,828)  0.4% (      229,406)
  Lahan Terbuka     6.4% (   43,535,357)  0.1% (       34,035)  0.0% (       24,624)
  Sawit             5.7% (   38,959,518)  0.0% (       11,751)  0.0% (        6,696)
  Pertanian Lain   12.9% (   87,846,593)  0.0% (        6,764)  0.0% (        1,300)
  Tambang           7.8% (   53,099,012)  0.0% (          121)  0.0% (           24)
  Permukiman       11.9% (   81,230,976)  0.0% (        5,177)  0.0% (        5,980)
  TOTAL piksel               683,540,480            62,259,200            62,324,736


## Bagian Training — Attention U-Net (SCSE) + ResNet-50

Resep identik konfigurasi proyek (`configs/default.yaml`): loss `focal_tversky`, median-frequency
class weights, weighted sampler kelas langka, AMP, warmup→cosine LR, transfer-learning 2-tahap
(freeze encoder beberapa epoch awal), grad-clip, resume dari checkpoint. Artefak → `MODEL_DIR`.

In [18]:
# === Worker tuning ===
import os
N_WORKERS = max(2, min(8, (os.cpu_count() or 2) - 1))
print(f"os.cpu_count()={os.cpu_count()} -> num_workers={N_WORKERS} "
      "(persistent_workers=True, pin_memory di build DataLoader).")

os.cpu_count()=2 -> num_workers=2 (persistent_workers=True, pin_memory di build DataLoader).


In [19]:
# === Build DataLoaders + model Attention U-Net (SCSE) + loss ===
from forestwatch.data import build_dataloaders_from_files
from forestwatch.model.architecture import build_unet, count_parameters
from forestwatch.model.losses import make_loss_fn

MODEL_KEY  = "model_1_attention_unet"
MODEL_ARCH = dict(architecture="unet_scse", encoder_name="resnet50")
CKPT_PATH  = MODEL_DIR / 'best_model.pt'

use_sampler = cfg['training'].get('use_weighted_sampler', True)
train_loader, val_loader, test_loader = build_dataloaders_from_files(
    final_train_files, val_files, test_files,
    batch_size=cfg['training']['batch_size'], num_workers=N_WORKERS,
    augment_p=cfg['training']['augmentation'],
    class_weights=class_weights if use_sampler else None,
    sampler_cache=SAMPLER_CACHE,   # cache shared di folder wilayah
    sampler_keys=None,             # file lokal -> key parts[-3:] cocok dgn cache
    persistent_workers=True,
)
print(f"Train {len(train_loader.dataset)} | Val {len(val_loader.dataset)} | Test {len(test_loader.dataset)}")

model = build_unet(
    in_channels=cfg['model']['in_channels'], classes=cfg['model']['classes'],
    encoder_weights=cfg['model']['encoder_weights'], **MODEL_ARCH,
)
print(f"{MODEL_KEY}: {count_parameters(model):,} param trainable")

lc = cfg['training']['loss']
loss_fn = make_loss_fn(
    loss_type=lc['type'],
    class_weights=class_weights if cfg['training'].get('use_class_weights', True) else None,
    tversky_alpha=lc.get('tversky_alpha', 0.3), tversky_beta=lc.get('tversky_beta', 0.7),
    focal_gamma=lc.get('focal_gamma', 2.0),
)
print('Loss:', lc['type'])

Train 10430 | Val 950 | Test 951


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

model_1_attention_unet: 33,826,354 param trainable
Loss: focal_tversky


In [20]:
# === Training (transfer-learning 2-tahap + resume + grad-clip) — RE-RUNNABLE ===
import time
from forestwatch.training.trainer import TrainConfig, train

tcfg = TrainConfig(
    epochs=cfg['training']['epochs'], patience=cfg['training']['patience'],
    learning_rate=cfg['training']['learning_rate'], weight_decay=cfg['training']['weight_decay'],
    amp=cfg['training']['amp'], warmup_epochs=cfg['training'].get('warmup_epochs', 3),
    freeze_encoder_epochs=cfg['training'].get('freeze_encoder_epochs', 3),
    grad_clip=cfg['training'].get('grad_clip', 1.0), resume=cfg['training'].get('resume', True),
    seed=cfg['project']['seed'], ckpt_path=CKPT_PATH.as_posix(),
)
_t0 = time.time()
summary = train(model, train_loader, val_loader, loss_fn=loss_fn, cfg=tcfg)
train_minutes = round((time.time() - _t0) / 60, 1)
print(f"\nbest val mIoU = {summary['best_val_iou']:.4f} @ epoch {summary['best_epoch']} | {train_minutes} menit")
print("ckpt   :", summary['ckpt_path'])

2026-06-18 18:38:01 [INFO] forestwatch.training: Scheduler: LinearLR warmup 3 ep -> CosineAnnealingLR.
2026-06-18 18:38:08 [INFO] forestwatch.training: Resume dari /content/drive/MyDrive/Satria Data 3.0/Training_Merauke_Boven_Digoel/model_1_attention_unet/best_model_resume.pt -> lanjut epoch 8 (best mIoU=0.3146).
2026-06-18 18:38:08 [INFO] forestwatch.training: Mulai training: device=cpu, AMP=False, epochs=80 (mulai ep 8)


Epoch 08 [train]:   0%|          | 0/1303 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
                                                                   

2026-06-19 01:16:16 [INFO] forestwatch.training: Ep 08/80 | train_loss 0.4047 | val_loss 0.0350 | val_mIoU 0.2846 | 23888.2s
2026-06-19 01:16:16 [INFO] forestwatch.training:   IoU per-kelas: [0.9994, 0.6235, 0.009, 0.0247, 0.0612, 0.0, 0.2746]


KeyboardInterrupt: 

In [ ]:
# === Plot history -> MODEL_DIR ===
import matplotlib.pyplot as plt
from forestwatch.constants import CLASS_COLORS

hist = summary['history']; epochs = [h['epoch'] for h in hist]
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
axes[0].plot(epochs, [h['train_loss'] for h in hist], label='train', lw=2)
axes[0].plot(epochs, [h['val_loss'] for h in hist], label='val', lw=2)
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(epochs, [h['val_miou'] for h in hist], color='green', lw=2)
axes[1].axhline(0.60, color='orange', ls='--', label='min 0.60')
axes[1].axhline(0.75, color='red', ls='--', label='ideal 0.75')
axes[1].set_title('Val mIoU (makro)'); axes[1].set_ylim(0, 1); axes[1].legend(); axes[1].grid(alpha=0.3)
if hist and 'val_iou_per_class' in hist[-1]:
    for c in range(N_CLASSES):
        ys = [h.get('val_iou_per_class', [float('nan')]*N_CLASSES)[c] for h in hist]
        axes[2].plot(epochs, ys, color=CLASS_COLORS[c], label=CLASS_NAMES[c], lw=1.6)
    axes[2].set_title('Val IoU per-kelas'); axes[2].set_ylim(0, 1)
    axes[2].legend(fontsize=7, ncol=2); axes[2].grid(alpha=0.3)
else:
    axes[2].plot(epochs, [h['lr'] for h in hist], color='purple'); axes[2].set_title('LR')
fig.suptitle(f'{MODEL_KEY} — Merauke + Boven Digoel'); fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'training_curve.png', dpi=120, bbox_inches='tight'); plt.show()
save_json({'best_val_iou': summary.get('best_val_iou'), 'best_epoch': summary.get('best_epoch'),
           'history': hist}, MODEL_DIR / 'training_history.json')
print('Disimpan:', OUTPUT_DIR / 'training_curve.png', '+ training_history.json')

In [ ]:
# === Evaluasi TEST (holdout Merauke+BD) -> metrics.json + confusion_matrix.png ===
import numpy as np, torch
import matplotlib.pyplot as plt
from forestwatch.training.trainer import evaluate
from forestwatch.training.metrics import compute_confusion_matrix, metric_summary

model.load_state_dict(torch.load(CKPT_PATH, map_location='cpu'))
preds, targets = evaluate(model, test_loader)
cm = compute_confusion_matrix(preds, targets, n_classes=N_CLASSES)
metrics = metric_summary(cm, class_names=CLASS_NAMES)
print(f"OA={metrics['overall_accuracy']*100:.2f}% | mIoU={metrics['mean_iou']:.4f} | kappa={metrics['kappa']:.4f}")
for row in metrics['per_class']:
    print(f"  {row['class']:<16} IoU={row['iou']:.4f}  F1={row['f1']:.4f}")
save_json(metrics, MODEL_DIR / 'metrics.json')

cm_np = np.array(metrics['confusion_matrix']); cm_norm = cm_np / cm_np.sum(axis=1, keepdims=True).clip(1)
fig, ax = plt.subplots(figsize=(8, 6)); im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        ax.text(j, i, f'{cm_norm[i, j]:.2f}', ha='center', va='center', fontsize=9,
                color='white' if cm_norm[i, j] > 0.5 else 'black')
ax.set_xticks(range(N_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
ax.set_yticks(range(N_CLASSES)); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(f'Confusion — {MODEL_KEY} (Merauke+BD)')
fig.colorbar(im, ax=ax); fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=120, bbox_inches='tight'); plt.show()
print('Disimpan:', MODEL_DIR / 'metrics.json', '+ confusion_matrix.png')

In [ ]:
# === Evaluasi SUPLEMEN: kapabilitas model kenali Tambang scr umum (BUKAN murni Merauke+BD) ===
# tambang_supplementary_test bersumber dari region transfer established (Sangatta/Morowali/dll),
# BUKAN representasi prevalensi Tambang LOKAL di Merauke+BD (yang nyaris nol -- lihat VALIDASI
# di atas: holdout murni cuma 1 patch/24px Tambang, terlalu kecil utk IoU reliable). Metrik di
# bawah ini JANGAN dicampur dgn metrics.json holdout murni -- laporkan terpisah di esai dgn label
# jelas: 'kemampuan umum kenali Tambang', bukan 'akurasi deteksi Tambang di Merauke+BD'.
from torch.utils.data import DataLoader
from forestwatch.data import PapuaDataset

if tambang_supplementary_test:
    _tb_loader = DataLoader(
        PapuaDataset(tambang_supplementary_test, train=False),
        batch_size=cfg['training']['batch_size'], shuffle=False, num_workers=N_WORKERS,
    )
    _tb_preds, _tb_targets = evaluate(model, _tb_loader)
    _tb_cm = compute_confusion_matrix(_tb_preds, _tb_targets, n_classes=N_CLASSES)
    tambang_supp_metrics = metric_summary(_tb_cm, class_names=CLASS_NAMES)
    print(f"[SUPLEMEN Tambang -- {len(tambang_supplementary_test)} patch transfer] "
          f"mIoU={tambang_supp_metrics['mean_iou']:.4f}")
    for row in tambang_supp_metrics['per_class']:
        if row['class'] == 'Tambang':
            print(f"  Tambang IoU={row['iou']:.4f}  F1={row['f1']:.4f}  "
                  f"(n_patch={len(tambang_supplementary_test)}, sumber=transfer, BUKAN lokal Merauke+BD)")
    save_json(tambang_supp_metrics, OUTPUT_DIR / 'tambang_supplementary_metrics.json')
    print('Disimpan:', OUTPUT_DIR / 'tambang_supplementary_metrics.json')
else:
    print('tambang_supplementary_test kosong -- lewati evaluasi suplemen.')

In [ ]:
# === Visualisasi kualitatif prediksi (GAMBAR) -> OUTPUT_DIR ===
# Bands patch: [B2,B3,B4,B8,B11,B12] -> RGB = (B4,B3,B2) = indeks (2,1,0).
import numpy as np, torch
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from matplotlib.patches import Patch
from forestwatch.constants import CLASS_COLORS

N_SHOW = 6
cmap_arr = np.array([to_rgb(CLASS_COLORS[c]) for c in range(N_CLASSES)])

def _rgb(img):
    rgb = np.stack([img[2], img[1], img[0]], axis=-1)   # R,G,B dari B4,B3,B2
    return np.clip(rgb * 3.0, 0, 1)                       # stretch reflektans ~[0,1]

def _color(lab):
    return cmap_arr[lab]

imgs, labs = [], []
for xb, yb in test_loader:
    for k in range(xb.shape[0]):
        imgs.append(xb[k].numpy()); labs.append(yb[k].numpy())
        if len(imgs) >= N_SHOW:
            break
    if len(imgs) >= N_SHOW:
        break

model.eval()
device = next(model.parameters()).device
with torch.no_grad():
    batch = torch.stack([torch.from_numpy(im) for im in imgs]).float().to(device)
    pred = model(batch).argmax(1).cpu().numpy()

fig, axes = plt.subplots(N_SHOW, 3, figsize=(10, 3 * N_SHOW))
for r in range(N_SHOW):
    axes[r, 0].imshow(_rgb(imgs[r]));   axes[r, 0].set_title('Citra (RGB)' if r == 0 else '')
    axes[r, 1].imshow(_color(labs[r])); axes[r, 1].set_title('Label asli' if r == 0 else '')
    axes[r, 2].imshow(_color(pred[r])); axes[r, 2].set_title('Prediksi model' if r == 0 else '')
    for cc in range(3):
        axes[r, cc].axis('off')
legend = [Patch(color=cmap_arr[c], label=CLASS_NAMES[c]) for c in range(N_CLASSES)]
fig.legend(handles=legend, loc='lower center', ncol=4, fontsize=8)
fig.suptitle('Prediksi kualitatif — TEST holdout Merauke + Boven Digoel')
fig.tight_layout(rect=[0, 0.04, 1, 0.97])
fig.savefig(OUTPUT_DIR / 'prediksi_kualitatif.png', dpi=120, bbox_inches='tight'); plt.show()
print('Disimpan:', OUTPUT_DIR / 'prediksi_kualitatif.png')

In [ ]:
# === Ekspor ONNX + tulis summary.json ===
from forestwatch.model.architecture import export_to_onnx
export_to_onnx(model, MODEL_DIR / 'model.onnx', in_channels=cfg['model']['in_channels'],
               patch_size=cfg['inference']['patch_size'], opset_version=13)

save_json({
    'model_key': MODEL_KEY, **MODEL_ARCH, 'area': 'Merauke + Boven Digoel',
    'tile_whitelist': sorted(TILE_WHITELIST),
    'best_val_iou': summary['best_val_iou'], 'best_epoch': summary['best_epoch'],
    'test_mean_iou': metrics['mean_iou'], 'test_overall_accuracy': metrics['overall_accuracy'],
    'test_kappa': metrics['kappa'], 'per_class': metrics['per_class'],
    'n_parameters': count_parameters(model), 'train_minutes': train_minutes,
    'n_train': len(final_train_files), 'n_val': len(val_files), 'n_test': len(test_files),
    'epochs_cfg': cfg['training']['epochs'], 'batch_size': cfg['training']['batch_size'],
    'class_weights': [float(w) for w in class_weights],
}, MODEL_DIR / 'summary.json')
print(f"OK Model 1 (Merauke+BD) SELESAI. Semua artefak di: {MODEL_DIR}")

In [ ]:
# === Simpan model + manifest ke folder wilayah Merauke+BD ===
import shutil

# best_model.pt sudah ditulis trainer; salin sbg nama final yang jelas + manifest folder.
FINAL_PT = MODEL_DIR / 'model_1_attention_unet_merauke.pt'
shutil.copy(CKPT_PATH, FINAL_PT)

manifest = {
    'area': 'Merauke + Boven Digoel',
    'tile_whitelist': sorted(TILE_WHITELIST),
    'model_key': MODEL_KEY, **MODEL_ARCH,
    'test_mean_iou': metrics['mean_iou'],
    'test_overall_accuracy': metrics['overall_accuracy'],
    'test_kappa': metrics['kappa'],
    'artefak': {
        'bobot_pt': FINAL_PT.name,
        'bobot_terbaik': CKPT_PATH.name,
        'onnx': 'model.onnx',
        'metrics': 'metrics.json',
        'summary': 'summary.json',
        'history': 'training_history.json',
        'gambar': 'output/  (training_curve.png, confusion_matrix.png, prediksi_kualitatif.png)',
    },
}
save_json(manifest, MODEL_DIR / 'manifest.json')

print('Model & artefak tersimpan di:', MODEL_DIR)
print('Isi folder:')
for p in sorted(MODEL_DIR.rglob('*')):
    if p.is_file():
        print(f"  {p.relative_to(MODEL_DIR).as_posix():<42} {p.stat().st_size/1e6:>8.1f} MB")